In [ ]:
!pip install lightgbm

import pandas as pd
import numpy as np
import lightgbm as lgb

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DATA_DIR = "/content/drive/My Drive/Colab Notebooks/Order_Forecasting/Processed"

X_train = pd.read_parquet(f"{DATA_DIR}/X_train.parquet")
X_val   = pd.read_parquet(f"{DATA_DIR}/X_val.parquet")

y_train = pd.read_parquet(f"{DATA_DIR}/y_train.parquet")["target"]
y_val   = pd.read_parquet(f"{DATA_DIR}/y_val.parquet")["target"]

In [ ]:
model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    random_state=42
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(100, verbose=False)]
)

pred_log = model.predict(X_val, num_iteration=model.best_iteration_)
pred = np.expm1(pred_log)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.362351 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3016
[LightGBM] [Info] Number of data points in the train set: 2844072, number of used features: 24
[LightGBM] [Info] Start training from score 2.916455


In [ ]:
pd.DataFrame({"prediction": pred}).to_csv(
    f"{DATA_DIR}/pred_lightgbm.csv", index=False
)